# Module 10: exercises  (solutions)

Most exercises use `students.csv`. Random seeds and splits are fixed for you so results are repeatable.

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

## Exercise 1

**Split.** Use `train_test_split` to split `X` and `y` (3 numeric features predicting `exam_score`) into 75% train / 25% test with `random_state=0`.

In [ ]:
import math

import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import mean_squared_error, precision_score, r2_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

students = pd.read_csv(DATA / "students.csv")
feats = ["study_hours_week", "sleep_hours", "attendance_pct"]
X, y = students[feats], students["exam_score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

In [ ]:
assert X_train.shape == (150, 3) and X_test.shape == (50, 3)
assert len(y_train) == 150 and len(y_test) == 50
assert set(X_train.index).isdisjoint(X_test.index), "Train and test must not overlap"
print("✅ Correct!")

## Exercise 2

**Baseline vs model.** Compute the test-set RMSE of (a) the baseline that predicts the **training mean** for every student (`rmse_baseline`) and (b) a `LinearRegression` fitted on the training set (`rmse_model`).

In [ ]:
baseline_pred = np.full(len(y_test), y_train.mean())
rmse_baseline = math.sqrt(mean_squared_error(y_test, baseline_pred))

model = LinearRegression().fit(X_train, y_train)
rmse_model = math.sqrt(mean_squared_error(y_test, model.predict(X_test)))

In [ ]:
assert abs(rmse_baseline - 12.0) < 0.1, f"baseline RMSE = {rmse_baseline}"
assert abs(rmse_model - 6.53) < 0.1, f"model RMSE = {rmse_model}"
assert rmse_model < rmse_baseline
print(f"✅ Correct! The model cuts the typical error from {rmse_baseline:.1f} to {rmse_model:.1f} points.")

## Exercise 3

**Cross-validation.** With the 5-fold splitter `cv` below, compute the R² of a `LinearRegression` on all of `X, y` with `cross_val_score(..., scoring="r2")`.
Store the mean as `cv_mean` and the standard deviation across folds as `cv_std`.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(LinearRegression(), X, y, cv=cv, scoring="r2")
cv_mean = scores.mean()
cv_std = scores.std()

In [ ]:
assert abs(cv_mean - 0.611) < 0.01, f"mean = {cv_mean}"
assert abs(cv_std - 0.143) < 0.02, f"std = {cv_std}"
print(f"✅ Correct! R² = {cv_mean:.2f} ± {cv_std:.2f}: report the spread, not just the mean")

## Exercise 4

**Overfitting by degree.** The noisy sine data and a train/test split are given. For each degree in `degrees`, fit
`make_pipeline(PolynomialFeatures(d), StandardScaler(), LinearRegression())` on the training data and record the **test** MSE in the dictionary `test_mse` (degree → MSE).
Set `best_degree` to the degree with the lowest test MSE, and `train_mse_12` to the **training** MSE of the degree-12 model.

In [ ]:
rng = np.random.default_rng(10)
xs = np.sort(rng.uniform(0, 1, 30))
ys = np.sin(2 * np.pi * xs) + rng.normal(0, 0.3, 30)
x_tr, x_te, y_tr, y_te = train_test_split(xs.reshape(-1, 1), ys, test_size=0.4, random_state=1)

degrees = [1, 3, 5, 9, 12]
test_mse = {}
for d in degrees:
    m = make_pipeline(PolynomialFeatures(d), StandardScaler(), LinearRegression()).fit(x_tr, y_tr)
    test_mse[d] = mean_squared_error(y_te, m.predict(x_te))
    if d == 12:
        train_mse_12 = mean_squared_error(y_tr, m.predict(x_tr))

best_degree = min(test_mse, key=test_mse.get)

In [ ]:
assert set(test_mse) == set(degrees)
assert best_degree == 3, f"best degree by test MSE: {best_degree}"
assert train_mse_12 < test_mse[12] / 5, "Degree 12 fits the training data far better than the test data: that's overfitting"
print(f"✅ Correct! degree 12: train MSE {train_mse_12:.3f} vs test MSE {test_mse[12]:.3f}")

## Exercise 5

**Pipeline.** Build `pipe`, a pipeline that first standardises the features (`StandardScaler`) and then fits a `Ridge(alpha=10)` regression. Evaluate it with the same 5-fold `cv` and store the mean R² as `ridge_cv`.

In [ ]:
pipe = make_pipeline(StandardScaler(), Ridge(alpha=10))
ridge_cv = cross_val_score(pipe, X, y, cv=cv, scoring="r2").mean()

In [ ]:
names = [name for name, _ in pipe.steps]
assert names == ["standardscaler", "ridge"], names
assert pipe.steps[1][1].alpha == 10
assert abs(ridge_cv - 0.614) < 0.01, f"CV R² = {ridge_cv}"
print(f"✅ Correct! Ridge CV R² = {ridge_cv:.3f}")

## Exercise 6

**Classification by hand.** The code builds `high = score >= 75` and a stratified split. Fit a pipeline `StandardScaler → LogisticRegression` on the training data, predict the test classes, and compute the confusion matrix
counts `tn, fp, fn, tp`. Then compute `precision` and `recall` **from those counts** (no sklearn metric functions).

In [ ]:
high = (y >= 75).astype(int)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X, high, test_size=0.25, random_state=0, stratify=high)

clf = make_pipeline(StandardScaler(), LogisticRegression()).fit(Xc_train, yc_train)
pred = clf.predict(Xc_test)
tn, fp, fn, tp = confusion_matrix(yc_test, pred).ravel()
precision = tp / (tp + fp)
recall = tp / (tp + fn)

In [ ]:
assert tn + fp + fn + tp == 50
assert tp + fn == yc_test.sum(), "tp + fn is the number of real positives"
assert abs(precision - precision_score(yc_test, clf.predict(Xc_test))) < 1e-9
assert abs(recall - recall_score(yc_test, clf.predict(Xc_test))) < 1e-9
assert (tp + tn) / 50 > 0.85
print(f"✅ Correct! precision {precision:.2f}, recall {recall:.2f}")

## Exercise 7

**The accuracy trap.** Write `metrics_from_counts(tp, fp, fn, tn)` returning a dict with `accuracy`, `precision`, `recall` and `f1` (use `0.0` whenever a denominator would be zero).
Apply it to a fraud detector that **never flags anything**, on 1,000 transactions of which 10 are fraud: `tp=0, fp=0, fn=10, tn=990`. Store the result in `lazy`.

In [ ]:
def metrics_from_counts(tp, fp, fn, tn):
    accuracy = (tp + tn) / (tp + fp + fn + tn)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

lazy = metrics_from_counts(tp=0, fp=0, fn=10, tn=990)

In [ ]:
got = metrics_from_counts(8, 2, 2, 88)
for key, want in {"accuracy": 0.96, "precision": 0.8, "recall": 0.8, "f1": 0.8}.items():
    assert abs(got[key] - want) < 1e-9, f"{key}: got {got[key]}, expected {want}"
assert abs(lazy["accuracy"] - 0.99) < 1e-9 and lazy["recall"] == 0.0 and lazy["f1"] == 0.0
print("✅ Correct! 99% accurate, and it catches no fraud at all.")

## Exercise 8

**Thresholds.** Using the fitted `clf` from Exercise 6, get `proba` (the probability of class 1 for the test set). Write `recall_at(t)` and `flagged_at(t)` returning the recall and the number of flagged students when
predicting "high" for `proba >= t`. Also store `auc`, the ROC AUC on the test set.

In [ ]:
proba = clf.predict_proba(Xc_test)[:, 1]

def recall_at(t):
    return recall_score(yc_test, (proba >= t).astype(int))

def flagged_at(t):
    return int((proba >= t).sum())

auc = roc_auc_score(yc_test, proba)

In [ ]:
assert flagged_at(0.1) >= flagged_at(0.5) >= flagged_at(0.9), "Lower thresholds flag more students"
assert recall_at(0.1) >= recall_at(0.5) >= recall_at(0.9), "Lower thresholds give higher (or equal) recall"
assert 0.90 < auc <= 1.0, f"AUC = {auc}"
print(f"✅ Correct! AUC = {auc:.2f}. At t=0.1 recall is {recall_at(0.1):.2f} but {flagged_at(0.1)} students are flagged vs {flagged_at(0.5)} at t=0.5.")

## Exercise 9

**Chronological split.** Load the rides, drop rows with missing values, sort by `date`. Train a `LinearRegression` on `temp_c` and `is_weekend` using the **first 300 days**, test on the remaining days.
Store `n_test` and `rmse_time` (the test RMSE).

In [ ]:
rides = pd.read_csv(DATA / "daily_rides.csv", parse_dates=["date"])
rides = rides.dropna().sort_values("date").reset_index(drop=True)
cols = ["temp_c", "is_weekend"]
train, test = rides.iloc[:300], rides.iloc[300:]
m = LinearRegression().fit(train[cols], train["rides"])
n_test = len(test)
rmse_time = math.sqrt(mean_squared_error(test["rides"], m.predict(test[cols])))

In [ ]:
assert n_test == 60, f"{n_test} test days (366 days - 6 missing = 360 rows; first 300 train)"
assert abs(rmse_time - 108.7) < 3, f"RMSE = {rmse_time}"
print(f"✅ Correct! Predicting the 'future' 60 days is off by about {rmse_time:.0f} rides on average.")

## Exercise 10

**Spot the problem.** A colleague reports that their model predicts exam scores with **cross-validated R² = 1.000** after adding a column `grade_points` (the exam score ÷ 25, taken from the final transcript). What is the most likely explanation? Set `answer` to a letter.

* **a)** The model is a perfect predictor and should be deployed immediately.
* **b)** Target leakage: `grade_points` is computed from the thing being predicted, so it wouldn't be available for a new student.
* **c)** The sample is too small.
* **d)** Cross-validation always gives R² = 1.

In [ ]:
answer = "b"

In [ ]:
assert answer == "b", "A score that's too good to be true usually is: look for leakage"
print("✅ Correct!")